<a href="https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule (plain words):** Compare each page's average impressions in Feb 2026 vs March 2026 (both fully observed, past-only — no leakage). If impressions in March are lower than Feb AND March impressions are still meaningfully large (≥100), flag it as declining-with-demand: worth reviewing before it fades further.

**Score:** `score = avg_impressions_mar` (rank declining pages by how much visibility is still at stake)
**Reason code:** `declining_with_demand`
**Action label:** `review_for_refresh`

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
!pip install -q huggingface_hub duckdb
import duckdb, os
import pandas as pd
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

base = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
queue = con.sql(f"""
WITH feb AS (
    SELECT content_hash_id, AVG(gsc_impressions) AS avg_impressions_feb
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-02/*.parquet')
    GROUP BY content_hash_id
),
mar AS (
    SELECT content_hash_id, AVG(gsc_impressions) AS avg_impressions_mar,
           AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    m.content_hash_id,
    f.avg_impressions_feb,
    m.avg_impressions_mar,
    m.avg_position,
    CASE WHEN m.avg_impressions_mar < f.avg_impressions_feb AND m.avg_impressions_mar >= 100
         THEN 'declining_with_demand' ELSE NULL END AS reason_code,
    CASE WHEN m.avg_impressions_mar < f.avg_impressions_feb AND m.avg_impressions_mar >= 100
         THEN 'review_for_refresh' ELSE 'monitor' END AS action,
    m.avg_impressions_mar AS score
FROM mar m JOIN feb f USING (content_hash_id)
ORDER BY score DESC
""").df()

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
queue.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,avg_impressions_feb,avg_impressions_mar,avg_position,reason_code,action,score
0,content_eadb33b5df496f4a,3387.571429,19907.225806,2.383011,None,monitor,19907.225806
1,content_ec2e0346994fb5a5,4280.500000,7912.129032,2.854514,None,monitor,7912.129032
2,content_e8a52cf3d5988c07,5790.321429,7901.000000,15.008339,None,monitor,7901.000000
3,content_0e03de7680314cd5,1008.464286,7139.032258,2.675217,None,monitor,7139.032258
4,content_44f34c0a90047651,3222.250000,6851.741935,7.346909,None,monitor,6851.741935
5,content_7172a7fad43f0998,4364.285714,6640.870968,3.367835,None,monitor,6640.870968
6,content_e7b5dd4dff461ad2,5517.928571,6614.354839,4.544203,None,monitor,6614.354839
7,content_8d7d99f109e19aa2,656.785714,6564.419355,2.563756,None,monitor,6564.419355
8,content_f107e54b10b43725,5577.250000,6322.483871,3.186054,None,monitor,6322.483871
9,content_36e53e9c707674fc,3597.714286,6276.741935,32.766674,None,monitor,6276.741935


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = queue[queue["reason_code"] == "declining_with_demand"].head(20)
top20

,content_hash_id,avg_impressions_feb,avg_impressions_mar,avg_position,reason_code,action,score
14,content_512dbad65bd5ade9,5975.107143,4979.290323,3.019798,declining_with_demand,review_for_refresh,4979.290323
19,content_e241d6415ac9e534,5862.571429,4590.451613,3.276016,declining_with_demand,review_for_refresh,4590.451613
24,content_8e1334d6356668e3,7264.321429,4354.322581,4.545582,declining_with_demand,review_for_refresh,4354.322581
27,content_0ec90963d98b97a5,5364.800000,4204.451613,3.249691,declining_with_demand,review_for_refresh,4204.451613
28,content_00d4fdf6e48a2d38,4619.035714,4091.483871,5.305905,declining_with_demand,review_for_refresh,4091.483871
30,content_fec55986a1868d62,6926.928571,4002.419355,9.385150,declining_with_demand,review_for_refresh,4002.419355
41,content_6302b8bce0bb84cb,4000.250000,3605.290323,3.127178,declining_with_demand,review_for_refresh,3605.290323
47,content_b556f0bd87d6fcca,4150.214286,3405.645161,3.863966,declining_with_demand,review_for_refresh,3405.645161
51,content_21309e9a83c83653,3786.178571,3328.612903,4.972998,declining_with_demand,review_for_refresh,3328.612903
52,content_cf651123f1085418,4418.142857,3269.774194,6.318624,declining_with_demand,review_for_refresh,3269.774194


1. content X — action: review_for_refresh, reason: impressions dropped from Y (Feb) to Z (Mar), still ≥100. Would be wrong if: this page belongs to a client whose GSC tracking only started in March (partial-month artifact, not a real decline).
... (repeat for all 20, or 10 if the card scope says 10)

## 4. Weak picks + leakage check

Weakest pick(s): [call out any row where the "decline" is likely a partial-month or single-outlier-day artifact rather than a real trend — check by pulling daily-level rows for that content_hash_id if suspicious].

Leakage check: this rule only uses Feb and Mar 2026 data — both fully in the past relative to any decision made in April. No product flags (`health_score`, `priority_score`, `action_type`) were used — confirmed absent from the schema in ML-04's DESCRIBE. No future window touched.

In [ ]:
assert not {"health_score","priority_score","action_type"} & set(queue.columns)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.